# SQL Practice GeneratorClaude-powered SQL practice with PostgreSQL and MySQL sandbox.**Workflow:** pick dialect + question type → generate problem → fill diagnostic form → write SQL → Test / Run / Submit.

In [1]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Setup ──
import os, sys, json, uuid
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd

# Load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv(Path(os.getcwd()).parent / '.env')
except Exception:
    pass

# Reload module imports so edits to .py files take effect on cell re-run
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
for mod in ['sql_practice_utils', 'sandbox']:
    if mod in sys.modules:
        del sys.modules[mod]
import sql_practice_utils as spu
import sandbox as sbx

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
GEN_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'generated_problems')
SOLVED_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'solved')
SESSIONS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'sessions')
for d in [GEN_DIR, SOLVED_DIR, SESSIONS_DIR]:
    os.makedirs(d, exist_ok=True)

# Init Claude
claude_ready = spu.init_claude()

# Check sandboxes
pg_ok, pg_msg = sbx.check_postgres()
my_ok, my_msg = sbx.check_mysql()
print(f"PostgreSQL: {pg_msg}")
print(f"MySQL:      {my_msg}")
if not pg_ok and not my_ok:
    print("\nNo sandbox reachable. Run `docker compose up -d` from the project root.")

# Shared state across cells
STATE = {
    'problem': None,
    'hint_index': 0,
    'last_action': None,
}

from IPython.display import Javascript, display

# Globally let textareas keep their native keystrokes by stopping
# JupyterLab's keyboard manager from grabbing them:
# - Shift+Enter: would re-run the cell and wipe widget state.
# - Cmd/Ctrl+Z, Cmd/Ctrl+Shift+Z: textarea native undo/redo would otherwise
#   collide with Jupyter's "undo cell action" binding.
# Tab/Shift+Tab are NOT stopped here because each textarea has its own
# Tab-indent handler that runs at the target level and calls preventDefault
# + stopPropagation itself. Stopping Tab here would kill those handlers.
display(Javascript("""
  document.addEventListener('keydown', function(e) {
    var inField = (e.target.tagName === 'TEXTAREA' || e.target.tagName === 'INPUT');
    if (!inField) return;
    if (e.shiftKey && e.key === 'Enter') {
      e.preventDefault();
      e.stopPropagation();
    }
    var modifier = e.metaKey || e.ctrlKey;
    if (modifier && (e.key === 'z' || e.key === 'Z')) {
      // native textarea undo/redo — keep Jupyter out of it
      e.stopPropagation();
    }
  }, true);
"""))

# Auto-resize textareas to fit their content as the user types — applies to
# .diagnose-textarea and .sql-code-editor textareas. Listens for input events
# and grows the textarea height to match scrollHeight. Mirrors nb02 behavior.
display(Javascript("""
  (function() {
    function autoResize(ta) {
      var scrollTop = window.scrollY;
      ta.style.height = "auto";
      ta.style.height = (ta.scrollHeight + 2) + "px";
      window.scrollTo({ top: scrollTop });
    }
    function attach(ta) {
      if (ta.dataset.autoResizeAttached) return;
      ta.dataset.autoResizeAttached = "1";
      ta.style.overflow = "hidden";
      ta.addEventListener("input", function() { autoResize(ta); });
      setTimeout(function() { autoResize(ta); }, 50);
    }
    function poll() {
      document.querySelectorAll(
        ".diagnose-textarea textarea, .sql-code-editor textarea"
      ).forEach(attach);
    }
    poll();
    setInterval(poll, 1000);
  })();
"""))


Claude ready (claude-sonnet-4-5)
PostgreSQL: Postgres reachable
MySQL:      MySQL reachable


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 1. Pick a problem

Choose dialect, question type, and either generate a new problem or replay one you've solved.

In [2]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Problem Picker ──

dialect_dd = widgets.Dropdown(
    options=[('PostgreSQL', 'postgresql'), ('MySQL', 'mysql')],
    value='postgresql',
    description='Dialect:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

qtype_dd = widgets.Dropdown(
    options=[(v['label'], k) for k, v in spu.QUESTION_TYPES.items()],
    value='select_analytical',
    description='Type:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'),
)

source_radio = widgets.RadioButtons(
    options=[('New (generate)', 'new'), ('Solved (replay)', 'solved')],
    value='new',
    description='Source:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

solved_dd = widgets.Dropdown(
    options=[('— pick a saved problem —', None)],
    description='Saved:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='600px', display='none'),
)

generate_btn = widgets.Button(description='Generate Problem', button_style='primary',
                              layout=widgets.Layout(width='200px', height='34px'))
status_out = widgets.Output()
problem_out = widgets.Output()

def refresh_solved_list(*_):
    items = spu.list_problems(GEN_DIR, dialect=dialect_dd.value, qtype=qtype_dd.value)
    opts = [('— pick a saved problem —', None)] + [
        (f"{p['generated_at'][:16]} · {p['title']}", p['path']) for p in items
    ]
    solved_dd.options = opts
    solved_dd.value = None

def on_source_change(change):
    if change['new'] == 'solved':
        solved_dd.layout.display = 'flex'
        refresh_solved_list()
    else:
        solved_dd.layout.display = 'none'

source_radio.observe(on_source_change, names='value')
dialect_dd.observe(refresh_solved_list, names='value')
qtype_dd.observe(refresh_solved_list, names='value')

def render_problem(p):
    with problem_out:
        clear_output(wait=True)
        title = p.get('title', 'Untitled')
        prompt_html = spu.prompt_to_bullets(p.get('prompt', ''))
        schema_html = spu.schema_to_html(p.get('schema_ddl', ''))
        data_html = spu.insert_data_to_html(p.get('example_input_data', ''))
        ex_cols = p.get('example_output_columns', [])
        ex_rows = p.get('example_output_rows', [])
        ex_df = pd.DataFrame(ex_rows, columns=ex_cols)
        meta = p.get('_meta', {})
        html = f'''
        <div style="border:1px solid #d0d7de; border-radius:6px; padding:16px; background:#fafbfc;">
          <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{meta.get('dialect','')} · {meta.get('question_type','')} · id {meta.get('problem_id','')}</div>
          <h3 style="margin:0 0 12px;">{title}</h3>
          <h4 style="margin-top:6px;">Prompt</h4>
          {prompt_html}
          <h4 style="margin-top:18px;">Schema</h4>
          {schema_html}
          <h4 style="margin-top:18px;">Example input data</h4>
          {data_html}
          <h4 style="margin-top:18px;">Expected output (example data)</h4>
          {ex_df.to_html(index=False, classes='ex-out')}
        </div>
        '''
        display(HTML(html))

def _attempt_progress(attempt, total, last_error):
    with status_out:
        if last_error:
            print(f'Attempt {attempt-1}/{total} failed validation: {last_error[:200]}')
            print(f'Retrying (attempt {attempt}/{total}) ...')
        else:
            print(f'Generating new {qtype_dd.value} problem in {dialect_dd.value} (attempt {attempt}/{total}) ...')

def on_generate(b):
    with status_out:
        clear_output(wait=True)
        if source_radio.value == 'solved':
            path = solved_dd.value
            if not path:
                print('Pick a saved problem first.')
                return
            print('Loading saved problem ...')
            problem = spu.load_problem(path)
        else:
            problem = spu.generate_problem(
                qtype_dd.value, dialect_dd.value, on_attempt=_attempt_progress
            )
            if not problem:
                print('Generation failed validation. Click Generate to try again.')
                return
            saved = spu.save_problem(problem, GEN_DIR)
            print(f'Validated and saved: {os.path.basename(saved)}')
            attempts = problem.get('_meta', {}).get('validation_attempts', 1)
            print(f'(passed validation on attempt {attempts})')
        STATE['problem'] = problem
        STATE['hint_index'] = 0
        # Refresh the problem reminder in the SQL editor section if it's been created
        try:
            refresh_reminder()
        except NameError:
            pass
        # Load schema and example data into the sandbox for the user to start with
        try:
            sbx.reset(dialect_dd.value)
            sbx.execute_script(dialect_dd.value, problem.get('schema_ddl', ''))
            sbx.execute_script(dialect_dd.value, problem.get('example_input_data', ''))
            print('Sandbox loaded with example data.')
        except Exception as e:
            print(f'Sandbox load error: {e}')
        render_problem(problem)

generate_btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HBox([dialect_dd, qtype_dd]),
    source_radio,
    solved_dd,
    generate_btn,
    status_out,
    problem_out,
]))

## 2. Diagnose before coding

Fill out the form, then click Get Feedback to have your problem analysis graded. Skip if you want to jump straight to coding.

In [3]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Diagnostic Form ──

paraphrase_ta = widgets.Textarea(
    placeholder='Restate the prompt in your own words ...',
    layout=widgets.Layout(width='100%', min_height='120px'),
    description='Paraphrase:',
    style={'description_width': '110px'},
)
paraphrase_ta.add_class('diagnose-textarea')

input_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Single table', 'single_table'), ('Join 2+', 'join'),
             ('Union', 'union'), ('Procedural', 'procedural')],
    description='Input:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

output_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Fewer rows', 'fewer_rows'), ('Same rows', 'same_rows'),
             ('Single value', 'single_value'), ('State mutation', 'state_mutation'),
             ('Scalar return', 'scalar_return'), ('Table return', 'table_return')],
    description='Output shape:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

recipe_dd = widgets.Dropdown(
    options=[('— pick —', '')] + [(r, r) for r in spu.RECIPE_VOCAB],
    description='Recipe:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

moves_ta = widgets.Textarea(
    placeholder='Move 1: ...\nMove 2: ...',
    layout=widgets.Layout(width='100%', min_height='90px'),
    description='Moves:',
    style={'description_width': '110px'},
)
moves_ta.add_class('diagnose-textarea')

feedback_btn = widgets.Button(description='Get Feedback', button_style='info',
                              layout=widgets.Layout(width='180px', height='34px'))
feedback_out = widgets.Output()

def fmt_check(ok, label, msg):
    badge = ('<span style="color:#1a7f37; font-weight:600;">correct</span>' if ok
             else '<span style="color:#cf222e; font-weight:600;">rethink</span>')
    return f'<li><strong>{label}:</strong> {badge} — {msg}</li>'

def on_feedback(b):
    with feedback_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        answers = {
            'paraphrase': paraphrase_ta.value,
            'input_arrival': input_dd.value,
            'output_shape': output_dd.value,
            'recipe': recipe_dd.value,
            'composite_moves': moves_ta.value,
        }
        print('Asking Claude to grade ...')
        result = spu.grade_diagnostic(STATE['problem'], answers)
        clear_output(wait=True)
        if not result:
            print('Grading failed.')
            return
        html = '<div style="border:1px solid #d0d7de; border-radius:6px; padding:14px; background:#f6f8fa;">'
        html += '<h4 style="margin:0 0 10px;">Diagnostic Feedback</h4>'
        html += '<ul style="line-height:1.7; margin:0 0 0 18px;">'
        html += f'<li><strong>Paraphrase:</strong> {result.get("paraphrase_feedback","")}</li>'
        html += fmt_check(result.get('input_classification_correct', False), 'Input',
                          result.get('input_classification_feedback',''))
        html += fmt_check(result.get('output_shape_correct', False), 'Output shape',
                          result.get('output_shape_feedback',''))
        html += fmt_check(result.get('recipe_correct', False), 'Recipe',
                          result.get('recipe_feedback',''))
        html += f'<li><strong>Moves:</strong> {result.get("composite_moves_feedback","")}</li>'
        html += '</ul>'
        html += f'<p style="margin:10px 0 0; font-style:italic; color:#57606a;">{result.get("overall","")}</p>'
        html += '</div>'
        display(HTML(html))

feedback_btn.on_click(on_feedback)

display(widgets.VBox([paraphrase_ta, input_dd, output_dd, recipe_dd, moves_ta, feedback_btn, feedback_out]))

# --- Diagnostic textareas: dracula theme + Tab/Shift+Tab indent handling ---
# Apply the same dark theme as the SQL editor and intercept Tab/Shift+Tab so
# they indent/outdent inside the textarea instead of triggering JupyterLab's
# tooltip or moving focus between cells.
display(HTML("""
<style>
.diagnose-textarea { width: 100% !important; }
.diagnose-textarea textarea {
  font: 14px/1.5 ui-monospace, Consolas, Menlo, monospace !important;
  tab-size: 4;
  -moz-tab-size: 4;
  border: 1px solid #44475a !important;
  border-radius: 6px !important;
  outline: none !important;
  padding: 8px 10px !important;
  background: #282a36 !important;
  color: #f8f8f2 !important;
  caret-color: #f8f8f2 !important;
  resize: vertical;
  min-height: 90px;
  width: 100% !important;
  box-sizing: border-box;
}
.diagnose-textarea textarea::placeholder { color: #6272a4 !important; }
.diagnose-textarea textarea::selection { background: #44475a !important; }
.diagnose-textarea textarea:focus { border-color: #bd93f9 !important; }
</style>
"""))

display(Javascript(r"""
(function () {
  function setupOne(ta) {
    if (ta.dataset.diagEnhanced) return;
    ta.dataset.diagEnhanced = '1';
    ta.addEventListener('keydown', function (e) {
      if (e.shiftKey && e.key === 'Enter') {
        e.preventDefault(); e.stopPropagation();
        var s = ta.selectionStart, en = ta.selectionEnd, v = ta.value;
        ta.value = v.slice(0, s) + '\n' + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + 1;
        ta.dispatchEvent(new Event('input', { bubbles: true }));
        return;
      }
      if (e.key !== 'Tab') return;
      e.preventDefault();
      e.stopPropagation();
      var s = ta.selectionStart, en = ta.selectionEnd;
      var v = ta.value;
      var indent = '    ';
      if (s === en && !e.shiftKey) {
        ta.value = v.slice(0, s) + indent + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + indent.length;
      } else {
        var before = v.slice(0, s);
        var lineStart = before.lastIndexOf('\n') + 1;
        var block = v.slice(lineStart, en);
        var lines = block.split('\n');
        var newLines;
        if (e.shiftKey) {
          newLines = lines.map(function (l) {
            if (l.startsWith(indent)) return l.slice(indent.length);
            if (l.startsWith('\t')) return l.slice(1);
            return l;
          });
        } else {
          newLines = lines.map(function (l) { return indent + l; });
        }
        var indented = newLines.join('\n');
        ta.value = v.slice(0, lineStart) + indented + v.slice(en);
        ta.selectionStart = lineStart;
        ta.selectionEnd = lineStart + indented.length;
      }
      ta.dispatchEvent(new Event('input', { bubbles: true }));
    });
  }
  function poll() {
    document.querySelectorAll('.diagnose-textarea textarea').forEach(setupOne);
  }
  poll();
  setInterval(poll, 1000);
})();
"""))

<IPython.core.display.Javascript object>

## 3. Write your SQL

- **Test**: run against example data, see output (does not check correctness).
- **Run**: compare your output to the expected output for example data.
- **Submit**: run against hidden test data; passing saves to your solved bank.

In [4]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Code Editor ──

def _render_problem_reminder():
    p = STATE.get('problem')
    if not p:
        return widgets.HTML('<div style="color:#57606a; padding:10px;"><i>Generate a problem first.</i></div>')
    title = p.get('title', 'Untitled')
    prompt_html = spu.prompt_to_bullets(p.get('prompt', ''))
    schema_html = spu.schema_to_html(p.get('schema_ddl', ''))
    data_html = spu.insert_data_to_html(p.get('example_input_data', ''))
    ex_cols = p.get('example_output_columns', [])
    ex_rows = p.get('example_output_rows', [])
    ex_df = pd.DataFrame(ex_rows, columns=ex_cols)
    expected_html = ex_df.to_html(index=False, classes='ex-out')
    return widgets.HTML(f'''
    <div style="border:1px solid #d0d7de; border-radius:6px; padding:12px 16px; background:#fafbfc; margin-bottom:10px;">
      <div style="font-weight:600; margin-bottom:6px; font-size:14px;">{title}</div>
      {prompt_html}
      <details style="margin-top:6px;"><summary style="cursor:pointer; color:#0969da;">Schema</summary>
      <div style="margin-top:6px;">{schema_html}</div>
      </details>
      <details style="margin-top:6px;"><summary style="cursor:pointer; color:#0969da;">Example input data</summary>
      <div style="margin-top:6px;">{data_html}</div>
      </details>
      <details style="margin-top:6px;" open><summary style="cursor:pointer; color:#0969da;">Expected output (example data)</summary>
      <div style="margin-top:6px;">{expected_html}</div>
      </details>
    </div>
    ''')

reminder_box = widgets.VBox([_render_problem_reminder()])

def refresh_reminder():
    reminder_box.children = [_render_problem_reminder()]

code_ta = widgets.Textarea(
    value='/* notes */\n\n',
    placeholder='-- Write your SQL here',
    layout=widgets.Layout(width='100%', min_height='240px'),
)
code_ta.add_class('sql-code-editor')

test_btn = widgets.Button(description='Test', button_style='',
                          layout=widgets.Layout(width='110px'))
run_btn  = widgets.Button(description='Run', button_style='info',
                          layout=widgets.Layout(width='110px'))
submit_btn = widgets.Button(description='Submit', button_style='success',
                            layout=widgets.Layout(width='110px'))
hint_btn = widgets.Button(description='Hint', button_style='warning',
                          layout=widgets.Layout(width='110px'))
format_btn = widgets.Button(description='Format', button_style='',
                            layout=widgets.Layout(width='110px'),
                            tooltip='Re-indent SQL using sqlparse; uppercase keywords, align clauses')
code_ref_btn = widgets.Button(description='Code Reference', button_style='',
                              layout=widgets.Layout(width='150px'),
                              tooltip='Show a generic framework skeleton for this question type (NOT the answer to your specific problem)')
result_out = widgets.Output()
hint_out = widgets.Output()

def on_hint(b):
    with hint_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        idx = STATE.get('hint_index', 0)
        text = spu.get_hint(STATE['problem'], idx)
        STATE['hint_index'] = idx + 1
        total = len(STATE['problem'].get('hints', []))
        display(HTML(f'<div style="background:#fff8c5; border-left:4px solid #d4a72c; padding:10px 14px; border-radius:4px;"><strong>Hint {min(idx+1,total)}/{total}:</strong> {text}</div>'))

hint_btn.on_click(on_hint)

def on_format(b):
    """Re-format SQL using sqlparse's reindent_aligned mode.
    Gentler than reindent=True — keeps clauses on their own lines, aligned,
    instead of smashing everything together. Still uppercases keywords.
    If you don't like the result, click again or just edit by hand.
    """
    try:
        import sqlparse
    except ImportError:
        with hint_out:
            clear_output()
            print('Format requires sqlparse. Install once with: pip install sqlparse')
        return
    raw = code_ta.value or ''
    # sqlparse joins statements without blank lines; split first, format each,
    # then rejoin with a blank line so the trailing SELECT stays separate from
    # the DO block.
    parts = [p for p in sqlparse.split(raw) if p.strip()]
    formatted = [
        sqlparse.format(
            p,
            reindent_aligned=True,
            keyword_case='upper',
            identifier_case=None,
            strip_comments=False,
        ).strip()
        for p in parts
    ]
    code_ta.value = '\n\n'.join(formatted)
format_btn.on_click(on_format)

def on_code_reference(b):
    """Show a generic SQL framework skeleton for the current question type.
    The skeleton uses placeholder names; it is NOT the answer to this problem.
    """
    with hint_out:
        clear_output(wait=True)
        p = STATE.get('problem')
        if not p:
            print('Generate a problem first.')
            return
        meta = p.get('_meta', {})
        qtype = meta.get('question_type', '')
        ref = spu.get_code_reference(
            qtype,
            islands_flavor=meta.get('islands_flavor'),
            percentile_flavor=meta.get('percentile_flavor'),
        )
        flavor_label = ''
        if meta.get('islands_flavor'):
            flavor_label = f" · {meta.get('islands_flavor')}"
        elif meta.get('percentile_flavor'):
            flavor_label = f" · {meta.get('percentile_flavor')}"
        import html as _html
        display(HTML(
            f'<div style="border:1px solid #d0d7de; border-radius:6px; padding:10px 14px; '
            f'background:#f6f8fa; margin-top:6px;">'
            f'<div style="font-weight:600; margin-bottom:6px; font-size:13px; color:#0969da;">'
            f'Code Reference — {qtype}{flavor_label} (generic framework, not the answer)</div>'
            f'<pre style="margin:0; background:#282a36; color:#f8f8f2; padding:10px 12px; '
            f'border-radius:4px; font: 13px/1.5 ui-monospace, Consolas, Menlo, monospace; '
            f'overflow-x:auto; white-space:pre;">{_html.escape(ref)}</pre>'
            f'</div>'
        ))
code_ref_btn.on_click(on_code_reference)

def _current_dialect():
    return STATE.get('problem', {}).get('_meta', {}).get('dialect', 'postgresql')

def _load_example_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('example_input_data', ''))

def _load_test_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('test_data', ''))

def _show_df(df, label='Output'):
    if df is None:
        return
    if df.empty:
        display(HTML(f'<div style="color:#57606a;"><b>{label}:</b> empty result set.</div>'))
    else:
        display(HTML(f'<h4>{label}</h4>' + df.to_html(index=False)))

def on_test(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        _show_df(df, 'Test output (example data)')

def on_run(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'example')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'CORRECT on example data' if ok else 'MISMATCH on example data'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        _show_df(df, 'Your output')
        _show_df(expected, 'Expected output')

def on_submit(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_test_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error on hidden test data:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'test')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'PASS — saved to solved bank' if ok else 'FAIL on hidden test data'
        if ok:
            try:
                spu.save_solved(STATE['problem'], code_ta.value, SOLVED_DIR)
            except Exception as e:
                msg += f'\n(could not save solved record: {e})'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        if not ok:
            _show_df(df, 'Your output (hidden test data)')
            _show_df(expected, 'Expected output (hidden test data)')

test_btn.on_click(on_test)
run_btn.on_click(on_run)
submit_btn.on_click(on_submit)

display(widgets.VBox([reminder_box, code_ta, widgets.HBox([test_btn, run_btn, submit_btn, hint_btn, format_btn, code_ref_btn]), hint_out, result_out]))

# --- SQL editor: dark theme + line-numbers gutter + Tab handler ---
# Plain ipywidgets Textarea (reliable typing). A JS-injected gutter sits to the
# left and shows line numbers. Tab inserts 4 spaces; Shift+Tab outdents.
# CSS gives the textarea a dracula-style dark background. No CodeMirror — that
# kept fighting JupyterLab's keyboard manager and breaking input.
from IPython.display import Javascript

display(HTML("""
<style>
.sql-editor-container {
  display: flex;
  border: 1px solid #44475a;
  border-radius: 6px;
  overflow: hidden;
  background: #282a36;
  margin-top: 6px;
  width: 100% !important;
  box-sizing: border-box;
}
.sql-editor-container .line-gutter {
  background: #21222c;
  color: #6272a4;
  padding: 8px 10px;
  text-align: right;
  font: 13px/1.5 ui-monospace, Consolas, Menlo, monospace;
  user-select: none;
  white-space: pre;
  min-width: 36px;
  border-right: 1px solid #44475a;
  overflow: hidden;
}
.sql-editor-container .line-gutter-inner { will-change: transform; }
.sql-code-editor { width: 100% !important; }
.sql-code-editor textarea {
  font: 14px/1.5 ui-monospace, Consolas, Menlo, monospace !important;
  tab-size: 4;
  -moz-tab-size: 4;
  border: none !important;
  outline: none !important;
  padding: 8px 10px !important;
  margin: 0 !important;
  background: #282a36 !important;
  color: #f8f8f2 !important;
  caret-color: #f8f8f2 !important;
  flex: 1 1 auto;
  resize: vertical;
  min-height: 280px;
  width: 100% !important;
}
.sql-code-editor textarea::placeholder { color: #6272a4 !important; }
.sql-code-editor textarea::selection { background: #44475a !important; }
</style>
"""))

display(Javascript(r"""
(function () {
  function setupOne(ta) {
    if (ta.dataset.sqlEnhanced) return;
    ta.dataset.sqlEnhanced = '1';
    var parent = ta.parentNode;
    var wrap = document.createElement('div');
    wrap.className = 'sql-editor-container';
    var gutter = document.createElement('div');
    gutter.className = 'line-gutter';
    var gutterInner = document.createElement('div');
    gutterInner.className = 'line-gutter-inner';
    gutter.appendChild(gutterInner);
    parent.insertBefore(wrap, ta);
    wrap.appendChild(gutter);
    wrap.appendChild(ta);

    function refresh() {
      var n = (ta.value.match(/\n/g) || []).length + 1;
      var s = '';
      for (var i = 1; i <= n; i++) s += i + '\n';
      gutterInner.textContent = s;
    }
    refresh();
    ta.addEventListener('input', refresh);
    ta.addEventListener('scroll', function () {
      gutterInner.style.transform = 'translateY(' + (-ta.scrollTop) + 'px)';
    });

    ta.addEventListener('keydown', function (e) {
      if (e.shiftKey && e.key === 'Enter') {
        e.preventDefault(); e.stopPropagation();
        var s = ta.selectionStart, en = ta.selectionEnd, v = ta.value;
        ta.value = v.slice(0, s) + '\n' + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + 1;
        ta.dispatchEvent(new Event('input', { bubbles: true }));
        return;
      }
      if (e.key !== 'Tab') return;
      e.preventDefault();
      e.stopPropagation();
      var s = ta.selectionStart, en = ta.selectionEnd;
      var v = ta.value;
      var indent = '    ';
      if (s === en && !e.shiftKey) {
        ta.value = v.slice(0, s) + indent + v.slice(en);
        ta.selectionStart = ta.selectionEnd = s + indent.length;
      } else {
        var before = v.slice(0, s);
        var lineStart = before.lastIndexOf('\n') + 1;
        var block = v.slice(lineStart, en);
        var lines = block.split('\n');
        var newLines;
        if (e.shiftKey) {
          newLines = lines.map(function (l) {
            if (l.startsWith(indent)) return l.slice(indent.length);
            if (l.startsWith('\t')) return l.slice(1);
            return l;
          });
        } else {
          newLines = lines.map(function (l) { return indent + l; });
        }
        var indented = newLines.join('\n');
        ta.value = v.slice(0, lineStart) + indented + v.slice(en);
        ta.selectionStart = lineStart;
        ta.selectionEnd = lineStart + indented.length;
      }
      ta.dispatchEvent(new Event('input', { bubbles: true }));
      refresh();
    });
  }
  function poll() {
    document.querySelectorAll('.sql-code-editor textarea').forEach(setupOne);
  }
  poll();
  setInterval(poll, 1000);
})();
"""))


<IPython.core.display.Javascript object>

## 4. Next problem

Clear all fields and start over.

In [5]:
from IPython.display import HTML as _HTML_toggle, display as _display_toggle
_display_toggle(_HTML_toggle('<button class="cw-code-toggle" onclick="(function(btn){var cell = btn.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;var hidden = input.style.display === \'none\';input.style.display = hidden ? \'\' : \'none\';btn.textContent = hidden ? \'\\u25B2 Hide code\' : \'\\u25BC Show code\';})(this); return false;" style="background:#0969da; color:white; padding:5px 12px; border:none; border-radius:4px; cursor:pointer; font-size:12px; margin-bottom:8px;">▲ Hide code</button><script>(function(){var s = document.currentScript;if (!s) return;setTimeout(function(){var cell = s.closest(\'.jp-Cell, .cell\');if (!cell) return;var input = cell.querySelector(\'.jp-Cell-inputWrapper, .input\');if (!input) return;input.style.display = \'none\';var btn = cell.querySelector(\'.cw-code-toggle\');if (btn) btn.textContent = \'\\u25BC Show code\';}, 80);})();</script>'))

# ── Next Question ──
next_btn = widgets.Button(description='Next Question (clear all)', button_style='danger',
                          layout=widgets.Layout(width='240px', height='34px'))
next_out = widgets.Output()

def on_next(b):
    STATE['problem'] = None
    STATE['hint_index'] = 0
    code_ta.value = '/* notes */\n\n'
    paraphrase_ta.value = ''
    moves_ta.value = ''
    input_dd.value = ''
    output_dd.value = ''
    recipe_dd.value = ''
    try:
        refresh_reminder()
    except NameError:
        pass
    for area in [problem_out, feedback_out, result_out, hint_out, status_out, next_out]:
        try:
            with area:
                clear_output(wait=True)
        except Exception:
            pass
    with next_out:
        print('Cleared. Generate a new problem above.')

next_btn.on_click(on_next)
display(widgets.VBox([next_btn, next_out]))

---**Files written each run:**- `data/outputs/generated_problems/` — every generated problem (loadable via Source: Solved).- `data/outputs/solved/` — successful submissions with your solution code.**Reset the sandbox** when something gets weird: `docker compose down -v && docker compose up -d`.